In [1]:
import tensorflow as tf
import numpy as np
import sklearn as sk
import os

I0000 00:00:1776455438.079576    7686 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776455438.362416    7686 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776455439.657136    7686 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
data = np.loadtxt('stock_sma_14_50_curated.csv', delimiter=',')

data = data.T
np.random.shuffle(data)
data = data.T

print(data.shape)

X = data[:504,:]
Y = data[-1:,:]

In [ ]:
X = data[:504,:]
Y = data[-1:,:]

print(X.shape)
print(Y.shape)

print(np.mean(Y))
print(np.std(Y))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='relu'),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.Huber(delta=0.5),
    metrics=['mae']
)

model.summary()


In [ ]:
# Clip extreme outliers in Y so MSE isn't dominated by rare spikes
# (p99 ≈ 2.09, but max ≈ 19 — those few outliers monopolize the gradient)
y_lo = np.percentile(Y, 2)
y_hi = np.percentile(Y, 98)
print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
Y = np.clip(Y, y_lo, y_hi)

In [ ]:
train_ratio = 0.8
train_size = int(X.shape[1] * train_ratio)

X_train = X[:,:train_size].T
Y_train = Y[:,:train_size].T

X_val = X[:,train_size:].T
Y_val = Y[:,train_size:].T

print(X_train.shape)
print(X_val.shape)

In [ ]:
# Normalize features
#mean = X_train.mean(axis=0)
#std  = X_train.std(axis=0) + 1e-8
#X_train = (X_train - mean) / std
#X_val   = (X_val   - mean) / std

# Normalize Y — without this the output layer fights a non-zero offset
# and the model collapses to predicting the mean/median
y_mean = Y_train.mean()
y_std  = Y_train.std() + 1e-8
Y_train_norm = (Y_train - y_mean) / y_std
Y_val_norm   = (Y_val   - y_mean) / y_std
print(f"Y normalized: mean→0, std→1  (y_mean={y_mean:.4f}, y_std={y_std:.4f})")


In [ ]:
model.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)


In [2]:
import matplotlib.pyplot as plt

FIGURE_FOLDER = 'figures/'

def plot_acc_and_loss(history, title: str, save: bool = False):
    file_title = title.replace(' ', '-').lower()

    if save and not os.path.exists(FIGURE_FOLDER): os.mkdir(FIGURE_FOLDER)

    plt.title("Model and Validation MAE for " + title)
    xs = list(range(1, len(history.history['mae']) + 1))
    plt.plot(xs, history.history['mae'], label="Model MAE", color="Red")
    plt.plot(xs, history.history['val_mae'], label="Validation MAE", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()

    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-mae.png')
        plt.close()
    else: plt.show()

    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    
    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-loss.png')
        plt.close()
    else: plt.show()

def prcnt_within_tolerance(X, Y, tolerance):
    return np.count_nonzero(np.abs(X - Y) <= tolerance) / X.shape[0]

def r2_score(model, X, Y):
    return sk.metrics.r2_score(Y, model.predict(X))

# Difference Formula for real numbers
def mean_prcnt_diff(model, X, Y):
    abs_Y = np.abs(Y)
    abs_Pred = np.abs(model.predict(X))

    raw_diffs = (np.abs(abs_Pred - abs_Y) / ((abs_Pred + abs_Y) / 2))

    #print(raw_diffs[:10])

    return np.mean(raw_diffs[np.isfinite(raw_diffs)])

In [ ]:
plot_acc_and_loss(model.history, "Model MAE")

In [ ]:
print("Prediction:", model.predict(X_val[:10]) * y_std + y_mean)
print("Label:", Y_val[:10])

In [ ]:
modelMSE = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelMSE.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelMSE.summary()

In [ ]:
modelMSE.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=30,
                  verbose=1,
                  shuffle=True)

In [ ]:
plot_acc_and_loss(modelMSE.history, "MSE Model")

In [ ]:
print("Prediction:", modelMSE.predict(X_val[:15]))
print("Label:", Y_val[:15])

In [ ]:
#print("Hubard Within 5%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.05))
#print("Hubard Within 10%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.1))
#print("Hubard Within 10%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.2))

In [ ]:
y_std = Y.std()

print("MSE Within 0.125 std (Mean is 0.0987):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 8))
print("MSE Within 0.25 std (Mean is 0.1974):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 4))
print("MSE Within 0.5 std (Mean is 0.383):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 2))
print("MSE Within 1 std (Mean is 0.68):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std))
r2_score(modelMSE)

In [ ]:
modelLG = tf.keras.Sequential([
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(756,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelLG.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelLG.summary()

In [ ]:
modelLG.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
modelWD = tf.keras.Sequential([
    tf.keras.layers.Dense(1000, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(500, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelWD.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelWD.summary()

In [ ]:
modelWD.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
print("WD Within 0.125(Avg is 0.0987):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.125))
print("WD Within 0.25 (Avg is 0.1974):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.25))
print("WD Within 0.5 (Avg is 0.383):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.5))
print("WD Within 1 (Avg is 0.68):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 1))

In [ ]:
plot_acc_and_loss(modelWD.history, "WD Model")

In [ ]:
modelLSTM = tf.keras.Sequential([
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Reshape((252, 1)),
    tf.keras.layers.LSTM(252),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelLSTM.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelLSTM.summary()

In [ ]:
modelLSTM.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
print("LSTM Within 0.125(Avg is 0.0987):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.125))
print("LSTM Within 0.25 (Avg is 0.1974):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.25))
print("LSTM Within 0.5 (Avg is 0.383):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.5))
print("LSTM Within 1 (Avg is 0.68):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 1))

In [ ]:
plot_acc_and_loss(modelLSTM.history, "LSTM Model")

In [ ]:
def mean_quadratic_error(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    squared_error = tf.square(y_pred - y_true)
    return tf.reduce_mean(squared_error, axis=-1)


modelMQE = tf.keras.Sequential([
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(253,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(126, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelMQE.compile(
    optimizer='adam',
    loss=mean_quadratic_error,
    metrics=['mae']
)

modelMQE.summary()

In [ ]:
historyMQE = modelMQE.fit(
    X_train,
    Y_train_norm,
    validation_data=(X_val, Y_val_norm),
    batch_size=64,
    epochs=20,
    verbose=1,
    shuffle=True,
)

In [ ]:
plot_acc_and_loss(historyMQE, "Custom MQE Model")

print("Prediction:", modelMQE.predict(X_val[:15]) * y_std + y_mean)
print("Label:", Y_val[:15])

In [ ]:
RESULTS_FOLDER = 'results/'

def train_sma_mse(window: int, offset: int, log: bool = False):
    file = f'stock_sma_{window}_{offset}_curated.csv'

    data = np.loadtxt(file, delimiter=',')

    data = data.T
    np.random.shuffle(data)
    data = data.T

    X = data[:504,:]
    Y = data[-1:,:]

    y_lo = np.percentile(Y, 2)
    y_hi = np.percentile(Y, 98)
    
    if log: print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
    
    Y = np.clip(Y, y_lo, y_hi)

    train_ratio = 0.8
    train_size = int(X.shape[1] * train_ratio)

    X_train = X[:,:train_size].T
    Y_train = Y[:,:train_size].T

    X_val = X[:,train_size:].T
    Y_val = Y[:,train_size:].T

    modelMSE = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

    modelMSE.compile(
        optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae']
    )

    if log: modelMSE.summary()

    modelMSE.fit(X_train, Y_train,
        validation_data=(X_val, Y_val),
        batch_size=64, 
        epochs=30,
        verbose=log,
        shuffle=True)
    
    plot_acc_and_loss(modelMSE.history, f"MSE {window} {offset} Model", save=True)

    y_std = Y.std()
    y_mean = np.full(X_val.shape[0], Y.mean())
    y_mean = y_mean.reshape((X_val.shape[0], 1))

    val_preds = modelMSE.predict(X_val)

    very_close = prcnt_within_tolerance(val_preds, Y_val, y_std / 8) / prcnt_within_tolerance(y_mean, Y_val, y_std / 8)
    close = prcnt_within_tolerance(val_preds, Y_val, y_std / 4) / prcnt_within_tolerance(y_mean, Y_val, y_std / 4)
    moderate = prcnt_within_tolerance(val_preds, Y_val, y_std / 2) / prcnt_within_tolerance(y_mean, Y_val, y_std / 2)
    far = prcnt_within_tolerance(val_preds, Y_val, y_std) / prcnt_within_tolerance(y_mean, Y_val, y_std)
    val_r2 = r2_score(modelMSE, X_val, Y_val)
    total_r2 = r2_score(modelMSE, X.T, Y.T)
    prcnts_off = mean_prcnt_diff(modelMSE, X_val, Y_val)
    
    if not os.path.exists(RESULTS_FOLDER): os.mkdir(RESULTS_FOLDER)

    file_title = f'mse-{window}-{offset}-results.csv'

    with open(RESULTS_FOLDER + file_title, 'w') as f:
        f.write('very_close,close,moderate,far,val_r2,total_r2,prcnts_off\n')
        f.write(f'{very_close},{close},{moderate},{far},{val_r2},{total_r2},{prcnts_off}\n')

    return [ modelMSE, X_train, Y_train, X_val, Y_val ]


In [ ]:
train_sma_mse(14, 14)

In [ ]:
train_sma_mse(3, 3, normalized=True)

In [ ]:
train_sma_mse(50, 50)

In [ ]:
train_sma_mse(50, 14)

In [ ]:
interval_list = [3,7,15,31,63,126]

for window in interval_list:
    for offset_len in interval_list:
        train_sma_mse(window, offset_len)
        print(f"Completed window:{window}, offset:{offset_len}")

In [6]:
interval_list = [3,7,15,31,63,126]
for offset_len in interval_list:
        train_sma_mse(3, offset_len, normalized=True)
        print(f"Completed window:3, offset:{offset_len}")

E0000 00:00:1776455489.447198    7686 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 734us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 678us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 635us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step
Completed window:3, offset:3


/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 639us/step
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 2s 630us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 640us/step
Completed window:3, offset:7


/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 697us/step
621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 707us/step
3105/3105 ━━━━━━━━━━━━━━━━━━━━ 2s 696us/step
621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 721us/step
Completed window:3, offset:31


/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 810us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 680us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 656us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 626us/step
Completed window:3, offset:63


/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step
621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step
3105/3105 ━━━━━━━━━━━━━━━━━━━━ 2s 655us/step
621/621 ━━━━━━━━━━━━━━━━━━━━ 0s 642us/step
Completed window:3, offset:126


In [8]:
train_sma_mse(3, 126, normalized=True, log=True)

Clipping Y to [-4.2519, 8.2989]  (was [-26.0912, 592671.3317])


/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 504)            │       254,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 504)            │         2,016 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 504)            │       254,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 504)            │         2,016 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 504)            │       254,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 504)            │         2,016 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │           505 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770,113 (2.94 MB)

 Trainable params: 767,089 (2.93 MB)

 Non-trainable params: 3,024 (11.81 KB)

Epoch 1/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 0.9420 - mae: 0.7408 - val_loss: 0.7852 - val_mae: 0.6703
Epoch 2/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - loss: 0.8012 - mae: 0.6818 - val_loss: 0.8122 - val_mae: 0.6891
Epoch 3/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.7926 - mae: 0.6765 - val_loss: 0.7882 - val_mae: 0.6785
Epoch 4/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.7896 - mae: 0.6747 - val_loss: 0.7908 - val_mae: 0.6733
Epoch 5/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.7873 - mae: 0.6732 - val_loss: 0.8011 - val_mae: 0.6791
Epoch 6/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.7843 - mae: 0.6717 - val_loss: 1.0481 - val_mae: 0.7861
Epoch 7/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.7777 - mae: 0.6693 - val_loss: 0.8036 - val_mae: 0.6771
Epoch 8/30
1242/1242 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.7748 - mae: 0.6690 - val_loss: 0.8355 - val_mae: 0.7106
Epoch 9/30
1242/1242 ━━━━━━━━━━━

In [12]:
best_model, xtrain, ytrain, xval, yval = train_sma_mse(3, 3)

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 778us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 686us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 685us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 694us/step


In [30]:
RESULTS_FOLDER = 'results/'

def train_and_eval(label: str, X, Y, model, log: bool = False):
    y_lo = np.percentile(Y, 2)
    y_hi = np.percentile(Y, 98)
    
    if log: print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
    
    Y = np.clip(Y, y_lo, y_hi)

    train_ratio = 0.8
    train_size = int(X.shape[1] * train_ratio)

    X_train = X[:,:train_size].T
    Y_train = Y[:,:train_size].T

    X_val = X[:,train_size:].T
    Y_val = Y[:,train_size:].T

    model.fit(X_train, Y_train,
        validation_data=(X_val, Y_val),
        batch_size=64, 
        epochs=30,
        verbose=log,
        shuffle=True)
    
    plot_acc_and_loss(model.history, label, save=True)

    y_std = Y.std()
    y_mean = np.full(X_val.shape[0], Y.mean())
    y_mean = y_mean.reshape((X_val.shape[0], 1))

    val_preds = model.predict(X_val)

    very_close = prcnt_within_tolerance(val_preds, Y_val, y_std / 8) / prcnt_within_tolerance(y_mean, Y_val, y_std / 8)
    close = prcnt_within_tolerance(val_preds, Y_val, y_std / 4) / prcnt_within_tolerance(y_mean, Y_val, y_std / 4)
    moderate = prcnt_within_tolerance(val_preds, Y_val, y_std / 2) / prcnt_within_tolerance(y_mean, Y_val, y_std / 2)
    far = prcnt_within_tolerance(val_preds, Y_val, y_std) / prcnt_within_tolerance(y_mean, Y_val, y_std)
    val_r2 = r2_score(model, X_val, Y_val)
    total_r2 = r2_score(model, X.T, Y.T)
    prcnts_off = mean_prcnt_diff(model, X_val, Y_val)
    
    if not os.path.exists(RESULTS_FOLDER): os.mkdir(RESULTS_FOLDER)

    file_title = f'{label}-results.csv'

    with open(RESULTS_FOLDER + file_title, 'w') as f:
        f.write('very_close,close,moderate,far,val_r2,total_r2,prcnts_off\n')
        f.write(f'{very_close},{close},{moderate},{far},{val_r2},{total_r2},{prcnts_off}\n')

    return [ model, X_train, Y_train, X_val, Y_val ]


In [25]:
data = np.loadtxt('stock_sma_3_3_curated.csv', delimiter=',')

data = data.T
np.random.shuffle(data)
data = data.T

print(data.shape)

X = data[:504,:]
Y = data[-1:,:]

(505, 99451)


In [31]:
modelMSEtanh = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='tanh', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEtanh.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

modelMSErelu = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSErelu.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

In [ ]:
train_and_eval("MSE window:3 offset:3 tanh", X, Y, modelMSEtanh)
train_and_eval("MSE window:3 offset:3 relu", X, Y, modelMSErelu)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 600us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 613us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 643us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 689us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 619us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 648us/step


[<Sequential name=sequential_19, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,

In [56]:
modelMSEthin = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthin.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
train_and_eval("MSE 3 3 Thin", X, Y, modelMSEthin)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 1s 405us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step


[<Sequential name=sequential_31, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,

In [58]:
modelMSEthinconv = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Reshape((504,1)),
        tf.keras.layers.Conv1D(filters=32, kernel_size=10, padding='same', activation='linear'),
        tf.keras.layers.Conv1D(filters=10, kernel_size=3, padding='same', activation='linear'),
        tf.keras.layers.Flatten(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthinconv.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [59]:
train_and_eval("MSE Thin Conv", X, Y, modelMSEthinconv)

622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 847us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 791us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 757us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 780us/step


[<Sequential name=sequential_32, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,

In [60]:
modelMSEthinconvpool = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Reshape((504,1)),
        tf.keras.layers.Conv1D(filters=32, kernel_size=10, padding='same', activation='linear'),
        tf.keras.layers.MaxPool1D(pool_size=10),
        tf.keras.layers.Flatten(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthinconvpool.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
train_and_eval("MSE Thin Conv and Pool", X, Y, modelMSEthinconvpool)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 687us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 593us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 597us/step


[<Sequential name=sequential_33, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,

In [62]:
modelMSEthinconvnodbn = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        #tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Reshape((504,1)),
        tf.keras.layers.Conv1D(filters=32, kernel_size=10, padding='same', activation='linear'),
        tf.keras.layers.Conv1D(filters=10, kernel_size=3, padding='same', activation='linear'),
        tf.keras.layers.Flatten(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthinconvnodbn.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [63]:
train_and_eval("MSE Without Dropout and Batch Normalization", X, Y, modelMSEthinconvpool)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 605us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 622us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 608us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step


[<Sequential name=sequential_33, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,